# Selecting Rows: Masks, Ranges And Chains

**Version**: 1

**Last modified**: 2026-10-12

**Compatible with opengate-gate-tree version:** 0.5.0

**Author**: Mateusz Jakub Bała

## About

A tree becomes an answer only once rows are picked out of it: the hits of one
event, the ones in an energy window, the ones a Compton scattering made. The
package names those questions on the pandas view of the data, and leaves alone
the ones a line of pandas already asks.

This notebook goes through the ways of picking rows — a mask, a range, a
selector, a chain — on one small file, and shows how they combine.

## What will you learn from this notebook

- which questions pandas already answers, and which the package adds
- the two forms every filter comes in: a mask, and the values it selects
- how to say which ends of a range belong to it
- how to combine masks with `&` and `|` and cut the rows once
- how to select by a process name or by what a gamma was
- how to chain selections through the `gate` namespace, and call the same
  filters as functions

## Tutorial

Every step below is a line of text, the code that does what the line says, and
a word about the answer where the answer needs one.

In [1]:
from tools import environment, paths

print(environment.describe())

opengate-gate-tree: 0.5.0
python: 3.14.5
run on: 2026-10-12


The header of this notebook states the version it was written for; the line
above states the version it just ran against. If they differ, the notebook may
still be right — but nobody has checked it against what is installed.

The example file holds 500 hits of a `PositroniumSource` scene; what it holds
is described in [`../data/README.md`](../data/README.md). Importing the package
is also what registers the `gate` namespace on pandas columns and frames.

In [2]:
from opengate_gate_tree import GammaType, GateTree, in_range, read_tree

example_input = paths.input_example("positronium-source-all-variants.root")

frame = read_tree(
    example_input,
    GateTree.HITS,
    [
        "runID",
        "eventID",
        "trackID",
        "edep",
        "posX",
        "posY",
        "posZ",
        "processName",
        "gammaType",
    ],
).to_dataframe()
frame

,runID,eventID,trackID,edep,posX,posY,posZ,processName,gammaType
0,0,0,2,0.094538,-184.148819,2.103387,-78.308502,Compton,2
1,0,0,2,0.416461,-184.547928,2.548918,-78.674561,PhotoElectric,2
2,0,0,1,0.220403,186.769531,-2.133321,79.422951,Compton,2
3,0,0,1,0.290596,186.650757,-2.092664,80.146744,PhotoElectric,2
4,0,1,2,0.132729,167.915894,77.666138,88.527336,Compton,2
...,...,...,...,...,...,...,...,...,...
495,0,113,1,0.033188,-93.682663,-149.508942,117.026428,Compton,0
496,0,113,1,0.225968,-93.084274,-149.406082,116.207977,PhotoElectric,0
497,0,114,3,0.505492,176.949173,-103.687340,-25.002714,PhotoElectric,2
498,0,114,2,0.099978,-127.397911,123.540756,92.852348,PhotoElectric,2


### What Pandas Already Says

Comparing a column, `isin` and `between` are pandas, and the package does not
restate them. How many hits each of the four tracks left, how many the first
two left together, and how many deposited between 200 and 511 keV — `edep` is
in MeV, as GATE writes it:

In [3]:
(
    frame["trackID"].value_counts().sort_index().to_dict(),
    int(frame["trackID"].isin([1, 2]).sum()),
    int(frame["edep"].between(0.2, 0.511).sum()),
)

({1: 215, 2: 173, 3: 87, 4: 25}, 388, 295)

A filter of the package earns its place by naming something the data means:
a closed range, a shape in the geometry of the detector, the identity of an
event, the meaning of a code GATE wrote.

### Every Filter Comes In Two

The same question is asked in two ways. `is_in_range` answers with a mask, one
boolean per row; `in_range` answers with the values the mask selects:

In [4]:
energetic = frame["edep"].gate.is_in_range(0.2, 0.511)
energetic_values = frame["edep"].gate.in_range(0.2, 0.511)

len(energetic), int(energetic.sum()), len(energetic_values)

(500, 295, 295)

The mask has a row for every row of the frame; the values are the 295 that
passed, and the same 295 `between` counted above — `is_in_range` is
`between` under another name, kept because it pairs with `in_range` and
because the shapes are built from it. The selected values keep the index they
had, which is what traces a value back to the entry it came from:

In [5]:
energetic_values.head()

1    0.416461
2    0.220403
3    0.290596
7    0.273780
8    0.510999
Name: edep, dtype: float32

### Which Ends Belong

Both ends belong to a range unless `inclusive` says otherwise, in the
vocabulary of `between`. On a column of whole numbers the difference is
plain to see — the hits of tracks 1 and 2:

In [6]:
{
    inclusive: int(frame["trackID"].gate.is_in_range(1, 2, inclusive=inclusive).sum())
    for inclusive in ("both", "left", "right", "neither")
}

{'both': 388, 'left': 215, 'right': 173, 'neither': 0}

`"both"` takes the 215 hits of track 1 and the 173 of track 2; `"left"` and
`"right"` take one of them each; `"neither"` takes nothing, since no whole
number lies strictly between 1 and 2.

### Combining Masks

A mask keeps the length and the index of the frame it was built from, so the
masks of one frame always combine. A question with several parts is built as
several masks, combined, and the rows are cut once — here the photoelectric
hits that deposited between 200 and 511 keV, or any hit of track 4:

In [7]:
photoelectric = frame["processName"].gate.is_process("PhotoElectric")
fourth_track = frame["trackID"] == 4

selected = frame[(energetic & photoelectric) | fourth_track]
len(selected), int((energetic & photoelectric).sum()), int(fourth_track.sum())

(255, 248, 25)

The two parts overlap, which is why the rows number fewer than the two
counts added up: a hit of track 4 that is also photoelectric and energetic is
selected once.

### Text Columns

`processName` is text, and its selectors take the names as GATE writes them.
`is_process` gives the mask, `select_by_process` the names themselves:

In [8]:
frame["processName"].gate.select_by_process("Compton", "PhotoElectric").value_counts()

processName
PhotoElectric    310
Compton          190
Name: count, dtype: int64

The names are not checked against a list: GATE builds differ in the physics
lists they were compiled with, and a name the package had never heard of would
be refused for no reason. The price is that a misspelt name is not refused
either — it simply matches nothing:

In [9]:
int(frame["processName"].gate.is_process("compt").sum())

0

### What A Gamma Was

The branches of a `PositroniumSource` hold integers, and their selectors take
the members of the class that names them, as many as the question needs:

In [10]:
prompt = frame["gammaType"].gate.is_gamma_type(GammaType.PROMPT)
int(prompt.sum())

72

A selector is asked about one column and answers about it, so
`select_by_gamma_type` gives the values of `gammaType` — a column holding
nothing but 3, the integer of `PROMPT`. That is what to count. Rows are
selected with the mask:

In [11]:
prompt_values = frame["gammaType"].gate.select_by_gamma_type(GammaType.PROMPT)
prompt_rows = frame[prompt]

prompt_values.unique().tolist(), prompt_values.shape, prompt_rows.shape

([3], (72,), (72, 9))

### Chaining

Each method of the `gate` namespace answers with what it was given — a frame
for a frame, a column for a column — so selections chain. The energies, up to
511 keV, of the hits of event 5 of run 0 that lie within 205 mm of the
centre:

In [12]:
energies = (
    frame.gate.by_event(0, 5).gate.in_sphere((0, 0, 0), 205.0)["edep"].gate.in_range(0.0, 0.511)
)
energies

21    0.510999
22    0.000748
23    0.028056
Name: edep, dtype: float32

An event is named by **both** identifiers, `runID` and `eventID`: GATE numbers
events within a run, so there is no filter taking the event alone. The index
on the left is the one the rows had in the file, so the answer still says
which entries it came from. How many hits of that event were left behind at
each step:

In [13]:
event = frame.gate.by_event(0, 5)
near = event.gate.in_sphere((0, 0, 0), 205.0)
len(event), len(near), len(energies)

(4, 3, 3)

Every hit of this file lies between 200 and 264 mm from the centre, so a
sphere of 205 mm keeps the hits nearest to it:

In [14]:
import numpy as np

distance = np.sqrt(frame["posX"] ** 2 + frame["posY"] ** 2 + frame["posZ"] ** 2)
float(distance.min()), float(distance.max())

(200.00242614746094, 263.50732421875)

### Functions Instead Of Methods

Every method of the namespace calls the function of the same name and adds
nothing. The functions are exported from the package root, which is what to
reach for when a filter is passed around rather than called:

In [15]:
in_range(frame["edep"], 0.2, 0.511).equals(frame["edep"].gate.in_range(0.2, 0.511))

True

## Summary

pandas compares, `isin` and `between`; the package adds ranges, shapes, events
and the meanings of codes, each as a mask (`is_*`) and as the selection itself
(`in_*`, `by_*`, `select_by_*`). Masks of one frame combine with `&` and `|`,
selections chain through the `gate` namespace, and both keep the index of the
rows they came from.

## Bibliography

- [Filtering And Selecting](https://opengate-gate-tree.readthedocs.io/en/latest/guide/filtering.html)
  — every filter, the two forms it comes in, and the `gate` namespace
- [Event Identifiers](https://opengate-gate-tree.readthedocs.io/en/latest/guide/events.html)
  — why an event is named by its run as well
- [PositroniumSource Data](https://opengate-gate-tree.readthedocs.io/en/latest/guide/positronium.html)
  — what the values of `gammaType` mean
- [Units](https://opengate-gate-tree.readthedocs.io/en/latest/guide/units.html)
  — why `edep` is in MeV and positions in mm
- [`pandas.Series.between`](https://pandas.pydata.org/docs/reference/api/pandas.Series.between.html)
  — the vocabulary of `inclusive`

## Changelog

- v1: written.